# Data Preparation

Steps to take, based on the initial analysis:

- Columns to drop (columns that don't help answer the research questions): Sub-product, Sub-issue, Consumer complaint narrative, Company public response, ZIP code, Tags, Consumer consent provided?
- Standardize column names
- Drop rows where State is: nan, military addresses and US territories
- Handle missing values in Consumer disputed? column
- Correct data types - convert 'Date received' and 'Date sent to company' to Datetime
- Save cleaned dataset

In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/consumer_complaints.csv')

## Handle columns

In [2]:
df = df.drop(columns=['Sub-product', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'ZIP code', 'Tags', 'Consumer consent provided?'])

In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[\s\-]+', '_', regex=True)
    .str.replace(r'[^\w]', '', regex=True)
)

## Convert data types

In [4]:
df['date_received'] = pd.to_datetime(df['date_received'])
df['date_sent_to_company'] = pd.to_datetime(df['date_sent_to_company'])

## Handle rows

In [5]:
df = df.dropna(subset=['state'])

# Military Addresses
df = df[df['state'] != 'AA']
df = df[df['state'] != 'AE']
df = df[df['state'] != 'AP']

# US Territories
df = df[df['state'] != 'AS']
df = df[df['state'] != 'GU']
df = df[df['state'] != 'MP']
df = df[df['state'] != 'PR']
df = df[df['state'] != 'VI']

### Consumer disputed missing values

The consumer_disputed column contains approximately 14% missing values. This is not random, as the CFPB discontinued its consumer dispute option in April 2017.

Instead of removing rows with missing values, they will be perserved as another category - 'Not available'.

In [7]:
df['consumer_disputed'] = df['consumer_disputed'].fillna('Not available')

print(df['consumer_disputed'].value_counts())

consumer_disputed
No               613056
Yes              146851
Not available    131459
Name: count, dtype: int64


## Validate

In [ ]:
df.info()

<class 'pandas.DataFrame'>
Index: 891366 entries, 0 to 903982
Data columns (total 11 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   date_received                 891366 non-null  datetime64[us]
 1   product                       891366 non-null  str           
 2   issue                         891366 non-null  str           
 3   company                       891366 non-null  str           
 4   state                         891366 non-null  str           
 5   submitted_via                 891366 non-null  str           
 6   date_sent_to_company          891366 non-null  datetime64[us]
 7   company_response_to_consumer  891366 non-null  str           
 8   timely_response               891366 non-null  str           
 9   consumer_disputed             891366 non-null  str           
 10  complaint_id                  891366 non-null  int64         
dtypes: datetime64[us](2), int64(1

In [10]:
df.isna().sum()

date_received                   0
product                         0
issue                           0
company                         0
state                           0
submitted_via                   0
date_sent_to_company            0
company_response_to_consumer    0
timely_response                 0
consumer_disputed               0
complaint_id                    0
dtype: int64

## Conclusions

- Columns that don't help with the analysis were dropped.
- Column names were standardized to lower case snake_case mode.
- 'date_received' and 'date_send_to_company' were converted to Datetime type.
- Rows were dropped where 'state' is one of following values: nan, military addresses and US territories.
- Missing values in 'consumer_disputed' were preserved as 'Not available'.

In [11]:
df.to_csv('../data/processed/consumer_complaints.csv', index=False)